In [12]:


import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


In [ ]:

g = 9.81

P = dict(Hs=6.0, Tp=8.8, gamma=3.3, h=20.0, Trec=10800.0, dt=0.1, fmin=0.04, fmax=0.50, seed=2026)



def wavenumber(w, h):
    k = w**2 / g
    for _ in range(5000):
        k = w**2 / (g * np.tanh(k * h))
    return k


def jonswap(f, Hs, Tp, gamma):
    """JONSWAP as in Lecture 5, slide 12: E = A_gamma * E_PM * gamma^exp[...]"""
    fp = 1.0 / Tp
    E_pm = (5 / 16) * Hs**2 * fp**4 * f**-5 * np.exp(-1.25 * (fp / f)**4)
    sig = np.where(f <= fp, 0.07, 0.09)
    A_gamma = 1 - 0.287 * np.log(gamma)
    return A_gamma * E_pm * gamma**np.exp(-(f - fp)**2 / (2 * sig**2 * fp**2))



def build_linear(seed):
    Nt = int(round(P["Trec"] / P["dt"]))
    df = 1.0 / P["Trec"]
    Tp = P["Tp"]
    fp = 1 / Tp
    n = np.arange(int(np.ceil(P["fmin"] / df)), int(np.floor(P["fmax"] / df)) + 1)
    f = n * df
    S = jonswap(f, P["Hs"], Tp, P["gamma"])
    a = np.sqrt(2 * S * df)
    phi = np.random.default_rng(seed).uniform(0, 2 * np.pi, len(n))
    arr = np.zeros(Nt, complex); arr[n] = a * np.exp(1j * phi)
    eta1 = np.real(np.fft.ifft(arr) * Nt)                    # sum a_n cos(2 pi n m/Nt + phi_n), m = time index
    return eta1, dict(Tp=Tp, fp=fp, n=n, f=f, S=S, df=df, m0=np.sum(S) * df, Nt=Nt)


def second_order_narrowband(eta1, info):
    """eta22 = G kp A^2 cos 2theta with envelope A and phase theta from the Hilbert transform (eq. 16-17)."""
    Nt = len(eta1)
    X = np.fft.fft(eta1)
    Hf = np.zeros(Nt); Hf[0] = 1; Hf[1:Nt // 2] = 2; Hf[Nt // 2] = 1
    z = np.fft.ifft(X * Hf)                                   # analytic signal A exp(i theta)
    A, theta = np.abs(z), np.angle(z)
    wp = 2 * np.pi * info["fp"]
    kp = wavenumber(np.array([wp]), P["h"])[0]
    kh = kp * P["h"]
    G = np.cosh(kh) * (2 + np.cosh(2 * kh)) / (4 * np.sinh(kh)**3)
    return eta1 + G * kp * A**2 * np.cos(2 * theta), dict(kp=kp, kph=kh, G=G)


def zero_crossing_waves(eta, dt):
    """Wave = record between two successive zero up-crossings. Returns crest elevation, wave height H, period."""
    x = eta - eta.mean()
    up = np.where((x[:-1] < 0) & (x[1:] >= 0))[0] + 1
    seg = x[up[0]:up[-1]]
    starts = up[:-1] - up[0]
    crest = np.maximum.reduceat(seg, starts)
    trough = np.minimum.reduceat(seg, starts)
    return crest, crest - trough, np.diff(up) * dt


def exceedance(v):
    v = np.sort(v)[::-1]
    return v, np.arange(1, len(v) + 1) / (len(v) + 1)


def rayleigh_crest(z, Hs): return np.exp(-8 * (z / Hs)**2)
def rayleigh_height(h, Hs): return np.exp(-2 * (h / Hs)**2)


def rayleigh_comparison(eta, Hs, dt, label):
    c, H, T = zero_crossing_waves(eta, dt)
    N = len(c)
    out = dict(label=label, N=N, Tz=T.mean(), Hm0=4 * np.std(eta), Hs_ref=Hs,
               H13=np.sort(H)[::-1][:N // 3].mean(), Hrms=np.sqrt(np.mean(H**2)), Hmean=H.mean(),
               Hmax=H.max(), cmax=c.max(), c=c, H=H,
               trough_min=(eta - eta.mean()).min(), skew=np.mean((eta - eta.mean())**3) / np.std(eta)**3)
    out["crest_frac"] = {q: np.mean(c > q * Hs) for q in (0.5, 0.75, 1.0)}
    out["H_frac"] = {q: np.mean(H > q * Hs) for q in (1.0, 1.5, 2.0)}
    out["Hmax_ray"] = Hs * np.sqrt(np.log(N) / 2)
    out["cmax_ray"] = Hs * np.sqrt(np.log(N) / 8)
    return out


def main():
    eta1, info = build_linear(P["seed"])
    eta2, g2 = second_order_narrowband(eta1, info)
    dt = P["dt"]
    Hs = 4 * np.sqrt(info["m0"])          # Hm0 of the discretised input spectrum: the Rayleigh scale parameter
    lines = []
    def say(s=""):
        print(s); lines.append(s)


    say(f"Rebuilt record: Tp = {info['Tp']:.2f} s (from kp*h = {P['kph_target']}), N_components = {len(info['n'])}, "
        f"Hm0(spectrum) = {Hs:.3f} m, 4 std(eta1) = {4*np.std(eta1):.3f} m")
    Tm02 = np.sqrt(info["m0"] / (np.sum(info["f"]**2 * info["S"]) * info["df"]))
    say(f"Tm02 of the components = 1/{1/Tm02:.3f} Hz -> {Tm02:.2f} s;  second order: kp*h = {g2['kph']:.2f}, G = {g2['G']:.2f}")

    lin = rayleigh_comparison(eta1, Hs, dt, "linear")
    sec = rayleigh_comparison(eta2, Hs, dt, "linear + 2nd order")

    say("\n=== Individual waves (zero up-crossings), seed %d ===" % P["seed"])
    say(f"{'':34s}{'linear':>10s}{'+ 2nd order':>13s}{'Rayleigh':>11s}")
    say(f"{'number of waves N':34s}{lin['N']:10d}{sec['N']:13d}")
    say(f"{'Tz [s]':34s}{lin['Tz']:10.2f}{sec['Tz']:13.2f}")
    say(f"{'Hm0 = 4 std(eta) [m]':34s}{lin['Hm0']:10.2f}{sec['Hm0']:13.2f}{Hs:11.2f}")
    say(f"{'H1/3 [m]':34s}{lin['H13']:10.2f}{sec['H13']:13.2f}{Hs:11.2f}")
    say(f"{'H1/3 / Hm0':34s}{lin['H13']/Hs:10.3f}{sec['H13']/Hs:13.3f}{1.0:11.3f}")
    say(f"{'Hrms [m]':34s}{lin['Hrms']:10.2f}{sec['Hrms']:13.2f}{Hs/np.sqrt(2):11.2f}")
    say(f"{'mean H [m]':34s}{lin['Hmean']:10.2f}{sec['Hmean']:13.2f}{np.sqrt(np.pi)/2*Hs/np.sqrt(2):11.2f}")
    say(f"{'highest crest [m]':34s}{lin['cmax']:10.2f}{sec['cmax']:13.2f}{lin['cmax_ray']:11.2f}")
    say(f"{'deepest trough [m]':34s}{lin['trough_min']:10.2f}{sec['trough_min']:13.2f}")
    say(f"{'largest wave height Hmax [m]':34s}{lin['Hmax']:10.2f}{sec['Hmax']:13.2f}{lin['Hmax_ray']:11.2f}")
    say(f"{'Hmax / Hs':34s}{lin['Hmax']/Hs:10.2f}{sec['Hmax']/Hs:13.2f}{lin['Hmax_ray']/Hs:11.2f}")
    say(f"{'skewness of eta':34s}{lin['skew']:10.2f}{sec['skew']:13.2f}")
    say("\nFraction of waves with crest above ...")
    for q in (0.5, 0.75, 1.0):
        say(f"   {q:4.2f} Hs = {q*Hs:4.2f} m : linear {lin['crest_frac'][q]:.4f}   +2nd {sec['crest_frac'][q]:.4f}   Rayleigh {rayleigh_crest(q*Hs, Hs):.4f}")
    say("Fraction of waves with wave height above ...")
    for q in (1.0, 1.5, 2.0):
        say(f"   {q:4.2f} Hs = {q*Hs:4.2f} m : linear {lin['H_frac'][q]:.4f}   +2nd {sec['H_frac'][q]:.4f}   Rayleigh {rayleigh_height(q*Hs, Hs):.4f}")

    # ---- ensemble of 100 other seeds: how much does one 3 h record scatter around Rayleigh?
    cs, Hh, cmax, Hmx, ncr = [], [], [], [], []
    for sd in range(1, 101):
        e, _ = build_linear(sd)
        c_, H_, _ = zero_crossing_waves(e, dt)
        cs.append(c_); Hh.append(H_); cmax.append(c_.max()); Hmx.append(H_.max())
    cs, Hh = np.concatenate(cs), np.concatenate(Hh)
    say(f"\n=== Ensemble: 100 linear records (3 h each, {len(cs)} waves pooled) ===")
    say(f"   highest crest per record: mean {np.mean(cmax):.2f} m, 5-95 % range {np.percentile(cmax,5):.2f}-{np.percentile(cmax,95):.2f} m (Rayleigh estimate {lin['cmax_ray']:.2f} m)")
    say(f"   largest H per record:     mean {np.mean(Hmx):.2f} m, 5-95 % range {np.percentile(Hmx,5):.2f}-{np.percentile(Hmx,95):.2f} m (Rayleigh estimate {lin['Hmax_ray']:.2f} m)")
    say(f"   pooled H1/3/Hm0 = {np.sort(Hh)[::-1][:len(Hh)//3].mean()/Hs:.3f}; pooled mean H/Hs = {Hh.mean()/Hs:.3f} (Rayleigh 0.627); Hrms/Hs = {np.sqrt(np.mean(Hh**2))/Hs:.3f} (Rayleigh 0.707)")
    for q in (0.5, 0.75, 1.0):
        say(f"   crest > {q:4.2f} Hs: pooled {np.mean(cs > q*Hs):.2e}  Rayleigh {rayleigh_crest(q*Hs, Hs):.2e}")
    for q in (1.0, 1.5, 2.0):
        say(f"   H     > {q:4.2f} Hs: pooled {np.mean(Hh > q*Hs):.2e}  Rayleigh {rayleigh_height(q*Hs, Hs):.2e}")

    # ---- figure
    BLUE, ORANGE, INK, GRID = "#257de0", "#f47230", "#191919", "#e6e6e6"
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.8))
    pairs = ((lin["c"], sec["c"], rayleigh_crest, np.linspace(0, 1.2, 300), r"crest elevation $\eta_c$ / $H_s$", "crest elevations"),
             (lin["H"], sec["H"], rayleigh_height, np.linspace(0, 2.2, 300), r"wave height $H$ / $H_s$", "wave heights"))
    for a_, (vl, vs, fray, xr, xl, ttl) in zip(ax, pairs):
        a_.semilogy(xr, fray(xr * Hs, Hs), color=INK, ls="--", lw=1.8, label="Rayleigh")
        x, p = exceedance(vl); a_.semilogy(x / Hs, p, "o", ms=3, color=BLUE, mec="none", label=f"linear (N = {lin['N']})")
        x, p = exceedance(vs); a_.semilogy(x / Hs, p, "o", ms=3, color=ORANGE, mec="none", label="linear + 2nd order")
        a_.set(xlim=(0, xr[-1]), ylim=(5e-4, 1.2), xlabel=xl, ylabel="exceedance probability")
        a_.set_title(f"Exceedance of individual {ttl} (seed {P['seed']}, 3 h)", fontsize=10, loc="left")
        a_.grid(color=GRID, lw=0.7); a_.set_axisbelow(True)
        for s_ in ("top", "right"): a_.spines[s_].set_visible(False)
        a_.legend(frameon=False, fontsize=9, loc="upper right")
    plt.tight_layout(); plt.savefig("q5_rayleigh.png", dpi=150)
    open("q5_results.txt", "w").write("\n".join(lines))


if __name__ == "__main__":
    main()
    # To analyse your own arrays instead of the rebuilt record:
    #   out = rayleigh_comparison(eta_linear, Hs=4*np.std(eta_linear), dt=0.1, label="linear")

main()


Rebuilt record: Tp = 9.02 s (from kp*h = 1.19), N_components = 4968, Hm0(spectrum) = 5.994 m, 4 std(eta1) = 5.994 m
Tm02 of the components = 1/0.139 Hz -> 7.19 s;  second order: kp*h = 1.19, G = 1.01

=== Individual waves (zero up-crossings), seed 2026 ===
                                      linear  + 2nd order   Rayleigh
number of waves N                       1516         1515
Tz [s]                                  7.12         7.12
Hm0 = 4 std(eta) [m]                    5.99         6.10       5.99
H1/3 [m]                                5.78         5.82       5.99
H1/3 / Hm0                             0.964        0.970      1.000
Hrms [m]                                4.10         4.13       4.24
mean H [m]                              3.63         3.64       3.76
highest crest [m]                       6.02         8.19       5.74
deepest trough [m]                     -5.68        -3.92
largest wave height Hmax [m]           10.75        11.78      11.47
Hmax / Hs        